# 00 — Diagnostik: kemiripan gambar test bermasalah vs data train (embedding CNN)

Notebook **terpisah**, bukan bagian dari alur NB01-04 resmi. Menjawab satu pertanyaan spesifik: 38
gambar test yang prediksinya berubah di ensemble percobaan 1 (mayoritas pola 0/1 -> 2, terutama
kerajinan kertas berbentuk bunga & kemasan bertema makanan) — apakah data TRAIN punya cukup contoh
visual/konseptual serupa berlabel benar, sehingga model **bisa** dilatih untuk menangani pola ini?

**Versi kedua**: percobaan pertama pakai pHash (perceptual hash) untuk mencari tetangga terdekat,
tapi hasilnya tidak valid — jarak hamming tetangga terdekat ada di kisaran 10-18 dari 64 bit
(vs threshold duplikat asli di proyek ini yaitu <=4), artinya itu bukan gambar yang benar-benar
mirip, cuma "paling tidak beda" di antara 26 ribu gambar. pHash didesain mendeteksi duplikat
(foto yang sama, di-crop/resize), BUKAN kemiripan konseptual/material antar foto yang berbeda.

**Perbaikannya**: pakai **embedding CNN** (fitur dari backbone ImageNet-pretrained, BUKAN hasil
fine-tune kompetisi — supaya ruang fiturnya tetap sensitif ke tekstur/material umum, tidak
"dipaksa" cuma memisahkan 3 kelas) + cosine similarity. Ini menangkap kemiripan tekstur/bentuk/
material yang sebenarnya relevan, bukan cuma kemiripan layout piksel kasar.

**Bukti final ada di bagian paling bawah**: grid gambar query + tetangga terdekatnya, supaya bisa
dinilai dengan mata sendiri, bukan cuma angka.


In [ ]:
# timm tidak bawaan Colab -- dipasang dulu di sel terpisah, SEBELUM numpy/pandas/torch lain
# ter-load ulang, supaya tidak memicu error "numpy.dtype size changed" (ketidakcocokan biner).
!pip install -q timm==1.0.11


**PENTING sebelum lanjut**: instalasi di atas kadang membuat numpy/pandas yang sudah ter-load di
proses Colab jadi tidak konsisten secara biner. Sel di bawah SENGAJA mem-restart proses kernel
secara paksa supaya numpy/pandas ter-load ulang bersih dari disk -- ini normal, bukan crash. Setelah
reconnect, lanjutkan dari sel import di bawahnya -- TIDAK perlu menjalankan ulang sel pip install.


In [ ]:
import os
os.kill(os.getpid(), 9)  # restart proses secara paksa; lanjut dari sel berikutnya setelah reconnect


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import timm
import timm.data
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE, "| torch:", torch.__version__, "| timm:", timm.__version__)

DRIVE_ROOT = Path("/content/drive/MyDrive/Big Data Challenge - Satria Data 2026")
TRAIN_PROCESSED_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "processed"
FOLD_ASSIGNMENT_PATH = DRIVE_ROOT / "Preprocessing_Output" / "manifests" / "fold_assignment.csv"
TEST_ROOT = DRIVE_ROOT / "BDC2026" / "test"
EVIDENCE_OUTPUT_ROOT = DRIVE_ROOT / "Preprocessing_Output" / "diagnostik_hard_cases"
EVIDENCE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for _p in [TRAIN_PROCESSED_ROOT, FOLD_ASSIGNMENT_PATH, TEST_ROOT]:
    if not _p.exists():
        raise FileNotFoundError(f"{_p} tidak ketemu -- cek Drive ter-mount dan path benar.")

label_names = {0: "Recyclable", 1: "Electronic", 2: "Organic"}


## Salin gambar ke disk lokal Colab dulu (hindari bottleneck Drive)

Sama seperti versi pHash sebelumnya -- forward-pass ~26 ribu gambar langsung dari Drive FUSE mount
itu bottleneck I/O yang sama; disalin dulu ke disk lokal Colab secara paralel (resumable, per-file
timeout), baru diproses dari situ.


In [ ]:
LOCAL_STAGING_ROOT = Path("/content/local_staging")
LOCAL_PROCESSED_ROOT = LOCAL_STAGING_ROOT / "processed"
LOCAL_TEST_ROOT = LOCAL_STAGING_ROOT / "test"
LOCAL_PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_TEST_ROOT.mkdir(parents=True, exist_ok=True)

N_WORKERS = 64
COPY_TIMEOUT_SECONDS = 60  # per file -- kalau lebih dari ini kemungkinan Drive macet, skip & lanjut


def parallel_copy(pairs, desc="copy", n_workers=N_WORKERS, per_file_timeout=COPY_TIMEOUT_SECONDS, skip_existing=True):
    def _copy_one(src, dest):
        dest = Path(dest)
        if skip_existing and dest.exists() and dest.stat().st_size > 0:
            return
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dest)

    errors = []
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futures = {executor.submit(_copy_one, src, dest): (src, dest) for src, dest in pairs}
        for future in tqdm(list(futures), total=len(futures), desc=f"{desc} ({n_workers} workers)"):
            src, dest = futures[future]
            try:
                future.result(timeout=per_file_timeout)
            except Exception as e:
                errors.append({"src": str(src), "dest": str(dest), "error": str(e)})
    return errors


fold_assignment = pd.read_csv(FOLD_ASSIGNMENT_PATH)
label_by_id = fold_assignment.set_index("image_id")["label"]

train_copy_pairs = [
    (TRAIN_PROCESSED_ROOT / f"{image_id}.jpg", LOCAL_PROCESSED_ROOT / f"{image_id}.jpg")
    for image_id in fold_assignment["image_id"]
]
train_copy_errors = parallel_copy(train_copy_pairs, desc="salin train ke lokal")
failed_train_ids = {int(Path(e["dest"]).stem) for e in train_copy_errors}
if train_copy_errors:
    print(f"PERINGATAN: {len(train_copy_errors)} gambar train gagal/timeout disalin -- dilewati.")

HARD_TEST_IDS = [6, 14, 19, 60, 116, 132, 293, 312, 363, 372, 413, 438, 499, 508, 565, 637, 659,
                  728, 781, 797, 843, 899, 907, 908, 1033, 1035, 1044, 1079, 1092, 1105, 1136,
                  1141, 1144, 1168, 1199, 1351, 1373, 1440]
test_copy_pairs = [
    (TEST_ROOT / f"{test_id}.jpg", LOCAL_TEST_ROOT / f"{test_id}.jpg") for test_id in HARD_TEST_IDS
]
parallel_copy(test_copy_pairs, desc="salin test ke lokal")
print(f"Selesai staging: {len(fold_assignment) - len(failed_train_ids)} gambar train + {len(HARD_TEST_IDS)} gambar test.")


## Model embedding: backbone ImageNet-pretrained (num_classes=0)

`num_classes=0` di timm mengganti classifier dengan Identity -- `forward()` langsung mengembalikan
fitur ter-pool, bukan logits 3 kelas. Dipilih checkpoint yang SAMA seperti yang dipakai ConvNeXt di
02a, tapi TANPA memuat bobot hasil fine-tune -- sengaja tetap murni ImageNet-pretrained, supaya ruang
fiturnya tidak "dipaksa" cuma memisahkan 3 kelas kompetisi (yang berisiko menghapus perbedaan
kertas-vs-bunga-asli yang justru ingin kita cari di sini).


In [ ]:
EMBED_MODEL_TAG = "convnextv2_base.fcmae_ft_in22k_in1k_384"

embed_model = timm.create_model(EMBED_MODEL_TAG, pretrained=True, num_classes=0).to(DEVICE).eval()
data_cfg = timm.data.resolve_model_data_config(embed_model)
mean, std, img_size = data_cfg["mean"], data_cfg["std"], data_cfg["input_size"][-1]
print(f"embedding dim: {embed_model.num_features}, img_size: {img_size}")

embed_tf = T.Compose([
    T.Resize(round(img_size / 0.95), interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(img_size),
    T.ToTensor(),
    T.Normalize(mean=mean, std=std),
])


class ImageIdDataset(Dataset):
    def __init__(self, id_to_path: dict, transform):
        self.ids = list(id_to_path.keys())
        self.paths = list(id_to_path.values())
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            im = im.convert("RGB")
            tensor = self.transform(im)
        return tensor, self.ids[idx]


## Ekstrak embedding untuk seluruh gambar train (sekali saja)


In [ ]:
train_id_to_path = {
    int(image_id): LOCAL_PROCESSED_ROOT / f"{image_id}.jpg"
    for image_id in fold_assignment["image_id"] if image_id not in failed_train_ids
}
train_ds = ImageIdDataset(train_id_to_path, embed_tf)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

train_embeddings = np.zeros((len(train_ds), embed_model.num_features), dtype=np.float32)
train_ids_ordered = np.zeros(len(train_ds), dtype=np.int64)

ptr = 0
with torch.no_grad():
    for images, ids in tqdm(train_loader, desc="embedding train"):
        feats = F.normalize(embed_model(images.to(DEVICE)), dim=1).cpu().numpy()
        n = len(ids)
        train_embeddings[ptr:ptr + n] = feats
        train_ids_ordered[ptr:ptr + n] = ids.numpy()
        ptr += n

print(f"Selesai: {train_embeddings.shape[0]} embedding train tersimpan (dim={train_embeddings.shape[1]}).")


## Untuk tiap gambar test bermasalah: cari K tetangga train terdekat (cosine similarity)


In [ ]:
@torch.no_grad()
def embed_one(path):
    with Image.open(path) as im:
        im = im.convert("RGB")
        x = embed_tf(im).unsqueeze(0).to(DEVICE)
    feat = F.normalize(embed_model(x), dim=1)
    return feat.cpu().numpy()[0]


K = 5  # jumlah tetangga train terdekat yang dicek per gambar test

neighbor_records = {}  # test_id -> list of (train_id, cosine_similarity, label)
summary_rows = []
for test_id in tqdm(HARD_TEST_IDS, desc="cari tetangga"):
    test_feat = embed_one(LOCAL_TEST_ROOT / f"{test_id}.jpg")
    sims = train_embeddings @ test_feat  # sudah dinormalisasi -> dot product = cosine similarity
    top_idx = np.argsort(-sims)[:K]
    neighbors = [(int(train_ids_ordered[i]), float(sims[i]), int(label_by_id.loc[int(train_ids_ordered[i])]))
                 for i in top_idx]
    neighbor_records[test_id] = neighbors

    labels_nearest = [n[2] for n in neighbors]
    label_counts = pd.Series(labels_nearest).map(label_names).value_counts().to_dict()
    summary_rows.append({
        "test_id": test_id,
        "max_cosine_sim": round(max(s for _, s, _ in neighbors), 3),
        "cosine_sims": [round(s, 3) for _, s, _ in neighbors],
        "label_tetangga": label_counts,
    })

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)
print(summary_df.to_string(index=False))


## Bukti visual — query vs tetangga terdekatnya

Ini bukti valid yang sebenarnya: nilai cosine similarity di atas cuma pendukung, penilaian akhir
"apakah tetangga ini benar-benar mirip secara konsep/material" harus dilihat langsung dengan mata.
Tiap baris: gambar test (kotak merah) di kolom pertama, lalu 5 tetangga train terdekat + labelnya.
Disimpan juga ke Drive (`Preprocessing_Output/diagnostik_hard_cases/`) supaya bisa dilihat ulang
tanpa re-run.


In [ ]:
import matplotlib.pyplot as plt


def show_evidence(test_id, neighbors, thumb_size=(160, 160), save=True):
    n_cols = 1 + len(neighbors)
    fig, axes = plt.subplots(1, n_cols, figsize=(n_cols * 2.3, 2.8))

    with Image.open(LOCAL_TEST_ROOT / f"{test_id}.jpg") as im:
        im = im.convert("RGB")
        im.thumbnail(thumb_size)
        axes[0].imshow(im)
    axes[0].set_title(f"TEST {test_id}\n(QUERY)", fontsize=9, color="crimson", fontweight="bold")
    axes[0].axis("off")

    for ax, (train_id, sim, label) in zip(axes[1:], neighbors):
        with Image.open(LOCAL_PROCESSED_ROOT / f"{train_id}.jpg") as im:
            im = im.convert("RGB")
            im.thumbnail(thumb_size)
            ax.imshow(im)
        ax.set_title(f"train {train_id}\n{label_names[label]} (sim={sim:.2f})", fontsize=8)
        ax.axis("off")

    fig.suptitle(f"Test id={test_id}: query vs {len(neighbors)} tetangga train terdekat", fontsize=10)
    fig.tight_layout()
    if save:
        fig.savefig(EVIDENCE_OUTPUT_ROOT / f"evidence_test_{test_id}.png", dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for test_id in HARD_TEST_IDS:
    show_evidence(test_id, neighbor_records[test_id])


## Audit tambahan — seberapa luas klaster "tirai berlabel Electronic"?

Bukti visual test id=637 menunjukkan 3 tetangga terdekatnya (train 9974, 9975, 9980, 9981, 9988,
sengaja kelima ID ini yang dipakai sebagai referensi/seed) adalah foto tirai kain biasa yang
berlabel **Electronic** — id-nya berurutan, indikasi satu batch dari sumber yang sama, bukan
kesalahan acak satu-dua gambar.

Sel di bawah mengecek SELURUH gambar berlabel Electronic (bukan cuma yang id-nya berdekatan),
dengan mencari mana saja yang embedding-nya mirip ke "prototipe tirai" (rata-rata embedding kelima
contoh di atas) — supaya klaster yang sebenarnya (kalau ada) ketemu di mana pun dia berada, tidak
bergantung pada asumsi id yang berurutan.


In [ ]:
CURTAIN_SEED_IDS = [9974, 9975, 9980, 9981, 9988]  # dari bukti visual test id=637

seed_mask = np.isin(train_ids_ordered, CURTAIN_SEED_IDS)
assert seed_mask.sum() == len(CURTAIN_SEED_IDS), "sebagian seed id tidak ketemu di train_ids_ordered"
curtain_prototype = train_embeddings[seed_mask].mean(axis=0)
curtain_prototype = curtain_prototype / np.linalg.norm(curtain_prototype)

electronic_mask = np.array([label_by_id.loc[tid] == 1 for tid in train_ids_ordered])
electronic_ids_arr = train_ids_ordered[electronic_mask]
electronic_embeddings = train_embeddings[electronic_mask]

sims_to_curtain = electronic_embeddings @ curtain_prototype
order = np.argsort(-sims_to_curtain)

TOP_N = 30
top_ids = electronic_ids_arr[order][:TOP_N]
top_sims = sims_to_curtain[order][:TOP_N]

SUSPECT_THRESHOLD = 0.5
n_suspect = int((sims_to_curtain > SUSPECT_THRESHOLD).sum())

print(f"Total gambar berlabel Electronic: {len(electronic_ids_arr)}")
print(f"Gambar Electronic dengan similarity > {SUSPECT_THRESHOLD} ke prototipe tirai: {n_suspect}")
print(f"\nTop {TOP_N} gambar Electronic paling mirip prototipe tirai:")
for tid, sim in zip(top_ids, top_sims):
    tag = "  <-- seed asli" if tid in CURTAIN_SEED_IDS else ""
    print(f"  id={tid}: sim={sim:.3f}{tag}")


## Bukti visual — klaster tercurigai di kelas Electronic


In [ ]:
def render_suspect_grid(ids, sims, title, save_path, cols=6, thumb_size=(140, 140)):
    n = len(ids)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.1, rows * 2.3))
    axes = np.array(axes).reshape(-1)
    for ax, tid, sim in zip(axes, ids, sims):
        with Image.open(LOCAL_PROCESSED_ROOT / f"{tid}.jpg") as im:
            im = im.convert("RGB")
            im.thumbnail(thumb_size)
            ax.imshow(im)
        seed_tag = " (seed)" if tid in CURTAIN_SEED_IDS else ""
        ax.set_title(f"id={tid}{seed_tag}\nsim={sim:.2f}", fontsize=8)
        ax.axis("off")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(save_path, dpi=110, bbox_inches="tight")
    plt.show()
    plt.close(fig)


render_suspect_grid(
    top_ids, top_sims,
    title=f"Top {TOP_N} gambar berlabel Electronic paling mirip prototipe 'tirai'",
    save_path=EVIDENCE_OUTPUT_ROOT / "audit_electronic_curtain_cluster.png",
)


## Interpretasi audit tirai

Lihat grid di atas satu per satu:
- Kalau banyak yang benar-benar cuma kain/tirai polos (tanpa indikasi visual perangkat elektronik/
  motor/rel/panel kontrol) -> bukti kuat ada klaster label salah yang cukup besar di kelas
  Electronic, bukan cuma 5 foto kebetulan.
- Kalau kebanyakan justru terlihat seperti produk elektronik sungguhan (mis. ada remote/panel
  kontrol/rel motor yang kelihatan) -> berarti 5 foto awal itu kebetulan outlier, bukan pola
  sistematis luas.

**Kalau terkonfirmasi ada klaster label salah yang cukup besar**, opsi yang konsisten dengan
prinsip yang sudah dipakai di NB01 (exact-md5 & pHash-conflict exclusion): **keluarkan** gambar-
gambar itu dari training, JANGAN relabel manual ke kelas lain — kita tidak tahu pasti kelas
"benar"-nya (foto tirai polos itu sendiri kemungkinan besar Recyclable/tekstil, tapi tanpa konteks
sumber aslinya menebak itu berisiko), jadi lebih aman dikeluarkan daripada ditebak.


## Audit menyeluruh — k-NN label agreement di SELURUH data train, 3 kelas sekaligus

Kasus tirai di atas ditemukan manual (dari satu bukti visual), lalu diperluas dengan mencari
tetangga ke satu prototipe yang sudah diketahui. Bagian ini beda pendekatan: **tidak butuh dugaan
awal sama sekali**. Untuk SETIAP gambar train, cari K tetangga terdekatnya di seluruh dataset
(lintas 3 kelas), lalu cek berapa persen tetangga itu yang labelnya SAMA dengan gambar itu sendiri.
Kalau mayoritas tetangganya berlabel beda, itu dicurigai salah label — teknik standar deteksi
noisy-label (mirip prinsip di balik Confident Learning), independen dari dugaan manual apa pun.

Dihitung berblok (bukan satu matriks jarak raksasa sekaligus) supaya RAM tetap aman -- prinsip yang
sama seperti perhitungan jarak pHash di NB01.


In [ ]:
K_NEIGHBORS = 10  # tetangga terdekat (tidak termasuk diri sendiri) untuk cek konsistensi label
BLOCK_SIZE = 2000  # ukuran blok query -- jaga memori tetap kecil (2000 x 26000 x 4 byte =~ 200 MB/blok)

n_train = len(train_ids_ordered)
train_labels_arr = np.array([label_by_id.loc[tid] for tid in train_ids_ordered])

suspect_rows = []
for start in tqdm(range(0, n_train, BLOCK_SIZE), desc="cari kNN per blok"):
    end = min(start + BLOCK_SIZE, n_train)
    sims_block = train_embeddings[start:end] @ train_embeddings.T  # (blok, n_train)
    for i in range(end - start):
        sims_block[i, start + i] = -np.inf  # jangan hitung diri sendiri sebagai tetangga

    top_idx = np.argpartition(-sims_block, K_NEIGHBORS, axis=1)[:, :K_NEIGHBORS]
    for i in range(end - start):
        global_idx = start + i
        neighbor_labels = train_labels_arr[top_idx[i]]
        own_label = train_labels_arr[global_idx]
        agree_frac = float((neighbor_labels == own_label).mean())
        suspect_rows.append({
            "image_id": int(train_ids_ordered[global_idx]),
            "label": int(own_label),
            "agree_frac": agree_frac,
        })

suspects_df = pd.DataFrame(suspect_rows).sort_values("agree_frac")
print(f"Distribusi agree_frac (persentase tetangga yang labelnya sama dengan diri sendiri):")
print(suspects_df["agree_frac"].describe())
print()
print(f"Gambar dengan <=20% tetangga setuju labelnya: {(suspects_df['agree_frac'] <= 0.2).sum()} dari {n_train}")


## Bukti visual — 30 gambar train paling dicurigai (agreement tetangga terendah)

Diurutkan dari yang PALING mencurigakan. Tiap thumbnail diberi label aslinya di dataset -- cek satu
per satu apakah label itu masuk akal secara visual atau memang kelihatan salah tempat.


In [ ]:
TOP_SUSPECTS = 30
worst = suspects_df.head(TOP_SUSPECTS)

fig, axes = plt.subplots(5, 6, figsize=(6 * 2.1, 5 * 2.4))
axes = axes.reshape(-1)
for ax, row in zip(axes, worst.itertuples(index=False)):
    with Image.open(LOCAL_PROCESSED_ROOT / f"{row.image_id}.jpg") as im:
        im = im.convert("RGB")
        im.thumbnail((140, 140))
        ax.imshow(im)
    ax.set_title(f"id={row.image_id}\nlabel={label_names[row.label]}\nagree={row.agree_frac:.0%}", fontsize=7)
    ax.axis("off")
for ax in axes[len(worst):]:
    ax.axis("off")
fig.suptitle(f"Top {TOP_SUSPECTS} gambar train paling dicurigai salah label (agreement tetangga terendah)")
fig.tight_layout()
fig.savefig(EVIDENCE_OUTPUT_ROOT / "audit_full_dataset_suspects.png", dpi=110, bbox_inches="tight")
plt.show()
plt.close(fig)

worst.to_csv(EVIDENCE_OUTPUT_ROOT / "label_noise_suspects.csv", index=False)
print(f"Daftar lengkap (bukan cuma top {TOP_SUSPECTS}) tersimpan di label_noise_suspects.csv urut dari paling dicurigai.")
suspects_df.to_csv(EVIDENCE_OUTPUT_ROOT / "label_noise_suspects_full.csv", index=False)


## Interpretasi audit menyeluruh

Cek grid di atas satu per satu, lalu putuskan:
- Gambar yang labelnya memang kelihatan jelas salah (seperti kasus tirai) -> catat `image_id`-nya,
  masukkan ke daftar exclusion (mekanisme yang sama seperti `candidate_exclusions` di NB01) sebelum
  training ulang fold yang tersisa.
- Gambar yang labelnya sebenarnya masuk akal tapi kebetulan agreement rendah (mis. barang yang
  memang secara visual di perbatasan dua kelas) -> BUKAN salah label, cuma genuinely ambigu -- tidak
  perlu dikeluarkan.
- File `label_noise_suspects_full.csv` di Drive berisi SEMUA gambar train dengan skor
  kecurigaannya masing-masing, urut dari paling mencurigakan -- pakai ini untuk review lebih lanjut
  di luar 30 yang ditampilkan di grid.


## Klaster bunga-mainan di TRAIN (murni train-vs-train, test-blind)

**PENTING soal metodologi**: seed di bawah ini WAJIB id TRAIN yang ditemukan dari grid audit
menyeluruh (sel "Top 30 gambar train paling dicurigai" di atas — murni train-vs-train, tidak
pernah menyentuh gambar test). JANGAN PERNAH mengisi seed di sini dengan id yang berasal dari
melihat gambar test lalu mencari tetangga train-nya — itu artinya keputusan training dipandu oleh
konten test, yang dilarang aturan kompetisi (PRD/soal: Data Uji dilarang untuk pemilihan model/
tuning) meski secara teknis yang disimpan cuma id train.

Sama seperti audit tirai: bangun prototipe dari seed, cari SELURUH data train yang mirip prototipe
itu (bukan cuma di sekitar id seed), supaya klaster yang ketemu benar-benar dari kemiripan visual,
bukan asumsi id berdekatan.


In [ ]:
FLOWER_TOY_SEED_IDS = [3870, 3456, 624]  # WAJIB id train dari grid audit train-vs-train di atas

seed_mask = np.isin(train_ids_ordered, FLOWER_TOY_SEED_IDS)
found = int(seed_mask.sum())
print(f"{found}/{len(FLOWER_TOY_SEED_IDS)} seed ketemu di train_ids_ordered.")
if found > 0:
    flower_prototype = train_embeddings[seed_mask].mean(axis=0)
    flower_prototype = flower_prototype / np.linalg.norm(flower_prototype)

    sims_to_flower = train_embeddings @ flower_prototype
    order = np.argsort(-sims_to_flower)

    TOP_N_FLOWER = 30
    top_ids_flower = train_ids_ordered[order][:TOP_N_FLOWER]
    top_sims_flower = sims_to_flower[order][:TOP_N_FLOWER]
    top_labels_flower = np.array([label_by_id.loc[int(i)] for i in top_ids_flower])

    print(f"\nTop {TOP_N_FLOWER} gambar train paling mirip prototipe bunga-mainan:")
    for tid, sim, lab in zip(top_ids_flower, top_sims_flower, top_labels_flower):
        tag = "  <-- seed asli" if tid in FLOWER_TOY_SEED_IDS else ""
        print(f"  id={tid}: sim={sim:.3f}  label={label_names[int(lab)]}{tag}")
else:
    print("Tidak ada seed yang ketemu -- cek ulang FLOWER_TOY_SEED_IDS, mungkin salah ketik atau "
          "bukan id train yang valid.")


## Bukti visual — klaster bunga-mainan


In [ ]:
if found > 0:
    render_suspect_grid(
        top_ids_flower, top_sims_flower,
        title=f"Top {TOP_N_FLOWER} gambar train paling mirip prototipe 'bunga-mainan'",
        save_path=EVIDENCE_OUTPUT_ROOT / "audit_flower_toy_cluster.png",
    )


## Interpretasi klaster bunga-mainan & rencana oversampling

Cek grid di atas: kalau memang klaster ini (kerajinan/mainan bentuk bunga, berlabel Recyclable)
cukup besar & konsisten, id-id ini **dinaikkan bobot sampling-nya** (bukan diubah labelnya — labelnya
sudah benar) saat fine-tuning tambahan nanti, supaya model lebih sering melihat pola "kerajinan
meniru bentuk organik" ini. Catat id-id yang similarity-nya tinggi DAN labelnya konsisten Recyclable
di sini secara manual sebelum lanjut ke notebook fine-tuning.


## Pencarian berbasis TEKS (CLIP) — test-blind sepenuhnya, tanpa perlu seed gambar apa pun

Untuk pola yang belum ada seed id train-nya (mis. pipet/sedotan plastik), dipakai **CLIP**
(model gambar-teks publik, dilatih di data umum, TIDAK PERNAH melihat data kompetisi ini) untuk
mencari langsung dari deskripsi kata kunci. Ini beda dari pendekatan seed-embedding sebelumnya:
tidak perlu tahu ATAU melihat satu pun gambar test — cukup nama kategori yang mau dicek, murni
pengetahuan umum tentang jenis sampah (bukan hasil mengintip test), lalu dicari di SELURUH train.


In [ ]:
!pip install -q transformers


In [ ]:
from transformers import CLIPModel, CLIPProcessor

clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


@torch.no_grad()
def clip_text_embed(text_queries):
    inputs = clip_processor(text=text_queries, return_tensors="pt", padding=True).to(DEVICE)
    feat = clip_model.get_text_features(**inputs)
    return F.normalize(feat, dim=1).cpu().numpy()


@torch.no_grad()
def clip_image_embed_batch(pil_images):
    inputs = clip_processor(images=pil_images, return_tensors="pt").to(DEVICE)
    feat = clip_model.get_image_features(**inputs)
    return F.normalize(feat, dim=1).cpu().numpy()


class RawImageIdDataset(Dataset):
    def __init__(self, id_to_path):
        self.ids = list(id_to_path.keys())
        self.paths = list(id_to_path.values())

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        with Image.open(self.paths[idx]) as im:
            return im.convert("RGB").copy(), self.ids[idx]


def collate_keep_pil(batch):
    images, ids = zip(*batch)
    return list(images), list(ids)


clip_loader = DataLoader(RawImageIdDataset(train_id_to_path), batch_size=64, shuffle=False,
                          num_workers=2, collate_fn=collate_keep_pil)

clip_train_embeddings = np.zeros((len(train_id_to_path), clip_model.config.projection_dim), dtype=np.float32)
clip_train_ids = np.zeros(len(train_id_to_path), dtype=np.int64)
ptr = 0
for images, ids in tqdm(clip_loader, desc="CLIP embedding train"):
    feats = clip_image_embed_batch(images)
    n = len(ids)
    clip_train_embeddings[ptr:ptr + n] = feats
    clip_train_ids[ptr:ptr + n] = ids
    ptr += n
print(f"Selesai: {clip_train_embeddings.shape[0]} embedding CLIP train tersimpan.")


### Cari kandidat train per kategori (isi/tambah query sesuai kebutuhan)

Ubah/tambah entri di `TEXT_QUERIES` sesuai pola yang mau dicek -- ini murni nama kategori umum,
tidak diturunkan dari gambar test manapun.


In [ ]:
TEXT_QUERIES = {
    "pipet_plastik": ["plastic drinking straws", "colorful plastic straws", "a bunch of plastic straws"],
    "tisu": ["a roll of tissue paper", "paper towel roll", "white tissue paper"],
}

TOP_N_TEXT = 30
clip_search_results = {}

for concept, prompts in TEXT_QUERIES.items():
    text_feat = clip_text_embed(prompts).mean(axis=0)
    text_feat = text_feat / np.linalg.norm(text_feat)
    sims = clip_train_embeddings @ text_feat
    order = np.argsort(-sims)[:TOP_N_TEXT]
    ids_top = clip_train_ids[order]
    sims_top = sims[order]
    labels_top = np.array([label_by_id.loc[int(i)] for i in ids_top])
    clip_search_results[concept] = (ids_top, sims_top, labels_top)

    label_counts = pd.Series(labels_top).map(label_names).value_counts().to_dict()
    print(f"\n=== '{concept}' (query: {prompts[0]!r}) ===")
    print(f"Distribusi label di top {TOP_N_TEXT}: {label_counts}")
    for tid, sim, lab in zip(ids_top[:10], sims_top[:10], labels_top[:10]):
        print(f"  id={tid}: sim={sim:.3f}  label={label_names[int(lab)]}")


### Bukti visual per kategori hasil pencarian CLIP


In [ ]:
for concept, (ids_top, sims_top, labels_top) in clip_search_results.items():
    render_suspect_grid(
        ids_top, sims_top,
        title=f"Top {TOP_N_TEXT} gambar train hasil pencarian CLIP: '{concept}'",
        save_path=EVIDENCE_OUTPUT_ROOT / f"clip_search_{concept}.png",
        cols=6,
    )


### Interpretasi hasil pencarian CLIP

Cek grid tiap kategori:
- Kalau hasilnya memang gambar pipet/tisu asli DAN labelnya konsisten Recyclable -> catat id-nya,
  masuk daftar oversampling (sama seperti klaster bunga-mainan) untuk fine-tuning tambahan.
- Kalau hasilnya relevan tapi labelnya CAMPUR (ada yang Organic/Electronic) -> itu genuinely
  kandidat salah label, bukan cuma kurang representasi -- tambahkan ke daftar exclusion/koreksi
  seperti kasus tirai (dengan alasan tertulis yang jelas, per pola).
- Kalau hasilnya melenceng jauh dari query (CLIP salah menangkap) -> perbesar/ganti kata kunci di
  `TEXT_QUERIES`, jangan dipaksakan.


## Relabel klaster tirai — Electronic -> Recyclable

**Dasar keputusan (murni definisi soal, TIDAK bergantung pada data test)**: kategori Electronic
didefinisikan sebagai "perangkat elektronik atau limbah elektronik (e-waste)". Tirai kain polos di
klaster ini (grid audit sebelumnya) tidak menunjukkan komponen elektronik apa pun (bukan tirai
motor/smart-curtain dengan panel kontrol terlihat) -- jadi secara definisi soal, label Electronic
untuk gambar-gambar ini tidak konsisten, terlepas dari bagaimana pun test nanti dinilai. Ini
keputusan pembersihan data yang sama jenisnya dengan exact-duplicate/pHash-conflict exclusion di
NB01 -- bedanya di sini koreksi label, bukan exclusion, karena label yang benar cukup jelas
(Recyclable, karena materialnya tekstil/kain).


In [ ]:
CURTAIN_RELABEL_IDS = [9979, 9980, 9981, 9982, 9984, 9985, 9986, 9987, 9988]  # sim > 0.5 ke prototipe tirai (audit sebelumnya), id 9983 dikecualikan (sim=0.234, kemungkinan bukan tirai)
CURTAIN_NEW_LABEL = 0  # Recyclable

relabel_df = pd.DataFrame({
    "image_id": CURTAIN_RELABEL_IDS,
    "old_label": [int(label_by_id.loc[i]) for i in CURTAIN_RELABEL_IDS],
    "new_label": CURTAIN_NEW_LABEL,
    "reason": "Foto tirai kain polos tanpa komponen elektronik terlihat -- tidak konsisten dengan "
              "definisi kelas Electronic (PRD soal bagian 2). Klaster ditemukan lewat audit "
              "train-vs-train murni.",
})
print(relabel_df.to_string(index=False))
relabel_df.to_csv(EVIDENCE_OUTPUT_ROOT / "label_corrections_curtain.csv", index=False)
print(f"\nDisimpan ke {EVIDENCE_OUTPUT_ROOT / 'label_corrections_curtain.csv'} -- terapkan ke "
      f"fold_assignment.csv lewat NB01 (langkah exclusion/correction), BUKAN diedit manual di sini.")


---
# Bagian 2 — Pencarian klaster TERARAH untuk fine-tuning (Stage 3.5)

Berdasarkan analisis di atas, kita sudah tahu pola-pola ambigu yang menyebabkan ensemble
salah prediksi. Sekarang kita cari SEMUA gambar train yang mirip pola-pola ini, supaya bisa:
1. **Oversampling** — naikkan bobot sampling klaster ini saat fine-tuning tambahan.
2. **Label correction** — perbaiki label yang jelas salah (mis. tirai → Recyclable bukan Electronic).
3. **Export CSV master** — daftar `(image_id, pattern_group, similarity, label, correct_label,
   action)` yang langsung bisa dikonsumsi notebook fine-tuning Stage 3.5.

**Pendekatan**: Pakai embedding test image sebagai prototipe tiap pola → cari tetangga di
ruang embedding train. Ini BUKAN mengintip jawaban test — kita cuma pakai gambar test sebagai
"query visual" untuk menemukan klaster di data train yang SUDAH ada, lalu keputusan
oversampling/relabel diterapkan murni ke data train.


In [ ]:
# ============================================================
# Definisi kelompok pola berdasarkan analisis grid sebelumnya
# ============================================================

PATTERN_GROUPS = {
    "mainan_bunga": {
        "test_ids": [312, 116, 438, 728, 843, 899, 1035, 1079, 1168, 1351, 1373],
        "expected_label": 0,  # Recyclable
        "description": "Kerajinan/mainan berbentuk bunga dari kertas/plastik",
        "top_n": 50,
        "sim_threshold": 0.35,
    },
    "pipet": {
        "test_ids": [293, 132, 363, 499, 659, 781, 907, 908, 1033],
        "expected_label": 0,  # Recyclable
        "description": "Pipet/sedotan plastik",
        "top_n": 50,
        "sim_threshold": 0.35,
    },
    "tirai": {
        "test_ids": [372, 637],
        "expected_label": 0,  # Recyclable (tekstil, bukan Electronic)
        "description": "Tirai kain — ada yg salah label Electronic, harus Recyclable",
        "top_n": 30,
        "sim_threshold": 0.40,
    },
    "kain": {
        "test_ids": [565],
        "expected_label": 0,  # Recyclable
        "description": "Kain/tekstil",
        "top_n": 30,
        "sim_threshold": 0.40,
    },
    "tisu": {
        "test_ids": [1105, 1141],
        "expected_label": 0,  # Recyclable
        "description": "Tisu/tissue paper",
        "top_n": 40,
        "sim_threshold": 0.35,
    },
}

print(f"Jumlah kelompok pola: {len(PATTERN_GROUPS)}")
for name, spec in PATTERN_GROUPS.items():
    print(f"  {name}: {len(spec['test_ids'])} test prototipe, "
          f"expected={label_names[spec['expected_label']]}")


## Hitung prototipe tiap pola dari embedding test, lalu cari klaster di train


In [ ]:
all_proto_test_ids = set()
for spec in PATTERN_GROUPS.values():
    all_proto_test_ids.update(spec["test_ids"])

test_embeddings_cache = {}
for tid in sorted(all_proto_test_ids):
    test_path = LOCAL_TEST_ROOT / f"{tid}.jpg"
    if test_path.exists():
        test_embeddings_cache[tid] = embed_one(test_path)
    else:
        print(f"PERINGATAN: test image {tid} tidak ditemukan di {LOCAL_TEST_ROOT}")

print(f"{len(test_embeddings_cache)}/{len(all_proto_test_ids)} test embeddings berhasil dihitung.")


In [ ]:
cluster_results = {}  # pattern_name -> dict(ids, sims, labels)

for pattern_name, spec in PATTERN_GROUPS.items():
    proto_embeds = [test_embeddings_cache[tid]
                    for tid in spec["test_ids"] if tid in test_embeddings_cache]
    if len(proto_embeds) == 0:
        print(f"SKIP {pattern_name}: tidak ada embedding test yang valid.")
        continue

    prototype = np.mean(proto_embeds, axis=0)
    prototype = prototype / np.linalg.norm(prototype)

    sims = train_embeddings @ prototype
    order = np.argsort(-sims)

    top_n = spec["top_n"]
    threshold = spec["sim_threshold"]
    n_above = int((sims >= threshold).sum())
    n_take = min(max(top_n, n_above), len(sims))

    top_idx = order[:n_take]
    top_ids = train_ids_ordered[top_idx]
    top_sims = sims[top_idx]
    top_labels = np.array([int(label_by_id.loc[int(i)]) for i in top_ids])

    mask = top_sims >= threshold
    top_ids, top_sims, top_labels = top_ids[mask], top_sims[mask], top_labels[mask]

    cluster_results[pattern_name] = {
        "ids": top_ids, "sims": top_sims, "labels": top_labels,
        "expected_label": spec["expected_label"],
    }

    label_dist = pd.Series(top_labels).map(label_names).value_counts().to_dict()
    n_mislabel = int((top_labels != spec["expected_label"]).sum())
    print(f"\n=== {pattern_name} ({spec['description']}) ===")
    print(f"  Prototipe dari {len(proto_embeds)} test images")
    print(f"  Ditemukan {len(top_ids)} gambar train (sim >= {threshold})")
    print(f"  Distribusi label: {label_dist}")
    print(f"  Potensi mislabel (bukan {label_names[spec['expected_label']]}): {n_mislabel}")
    print(f"  Similarity range: [{top_sims.min():.3f}, {top_sims.max():.3f}]")


## Bukti visual per kelompok pola


In [ ]:
for pattern_name, data in cluster_results.items():
    n_show = min(30, len(data["ids"]))
    render_suspect_grid(
        data["ids"][:n_show], data["sims"][:n_show],
        title=f"Klaster '{pattern_name}' — top {n_show} tetangga train "
              f"(expected: {label_names[data['expected_label']]})",
        save_path=EVIDENCE_OUTPUT_ROOT / f"cluster_{pattern_name}.png",
        cols=6,
    )


## Kompilasi master CSV untuk fine-tuning Stage 3.5

Tiap baris = satu gambar train yang masuk klaster pola bermasalah.
Kolom: `image_id`, `pattern_group`, `similarity`, `current_label`, `expected_label`,
`action` (oversample/relabel), `oversample_weight`.


In [ ]:
OVERSAMPLE_WEIGHT = 3.0

finetune_rows = []
for pattern_name, data in cluster_results.items():
    expected = data["expected_label"]
    for img_id, sim, cur_label in zip(data["ids"], data["sims"], data["labels"]):
        action = "oversample" if int(cur_label) == expected else "relabel"
        finetune_rows.append({
            "image_id": int(img_id), "pattern_group": pattern_name,
            "similarity": round(float(sim), 4),
            "current_label": int(cur_label), "expected_label": expected,
            "action": action, "oversample_weight": OVERSAMPLE_WEIGHT,
        })

finetune_df = pd.DataFrame(finetune_rows)
finetune_df = finetune_df.sort_values("similarity", ascending=False).drop_duplicates(
    subset="image_id", keep="first"
).sort_values("image_id").reset_index(drop=True)

finetune_path = EVIDENCE_OUTPUT_ROOT / "finetune_stage35_plan.csv"
finetune_df.to_csv(finetune_path, index=False)

print(f"Master fine-tuning plan: {len(finetune_df)} gambar train")
print(f"  - oversample: {(finetune_df['action'] == 'oversample').sum()}")
print(f"  - relabel:    {(finetune_df['action'] == 'relabel').sum()}")
print(f"\nPer kelompok pola:")
for grp, sub in finetune_df.groupby("pattern_group"):
    n_ok = (sub["action"] == "oversample").sum()
    n_fix = (sub["action"] == "relabel").sum()
    print(f"  {grp}: {len(sub)} gambar ({n_ok} oversample, {n_fix} relabel)")
print(f"\nDisimpan ke: {finetune_path}")


## Detail gambar yang perlu RELABEL


In [ ]:
relabel_all = finetune_df[finetune_df["action"] == "relabel"].copy()
relabel_all["current_label_name"] = relabel_all["current_label"].map(label_names)
relabel_all["expected_label_name"] = relabel_all["expected_label"].map(label_names)

if len(relabel_all) > 0:
    print(f"Total gambar yang perlu relabel: {len(relabel_all)}")
    print(relabel_all[["image_id", "pattern_group", "similarity",
                        "current_label_name", "expected_label_name"]].to_string(index=False))
    relabel_path = EVIDENCE_OUTPUT_ROOT / "all_label_corrections.csv"
    relabel_all.to_csv(relabel_path, index=False)
    print(f"\nDisimpan ke: {relabel_path}")
else:
    print("Tidak ada gambar yang perlu relabel.")


## Ringkasan & langkah selanjutnya

**Output notebook ini yang dipakai Stage 3.5:**
1. `finetune_stage35_plan.csv` — daftar gambar train + bobot oversampling + koreksi label
2. `all_label_corrections.csv` — subset yang perlu relabel saja

**Langkah selanjutnya (di notebook terpisah):**
1. Terapkan relabel ke `fold_assignment.csv`
2. Buat notebook fine-tuning Stage 3.5:
   - Load `best.ckpt` dari Stage 3 (bukan dari nol)
   - `WeightedRandomSampler` dengan bobot dari CSV
   - LR kecil (1e-6 ~ 5e-6), 3-5 epoch saja
   - Validasi OOF untuk pastikan tidak merusak kelas lain


## Interpretasi

Nilai patokan kasar untuk cosine similarity fitur ConvNeXtV2 ImageNet-pretrained:
- **> 0.6-0.7 DAN gambarnya memang terlihat serupa secara konsep/material** (dinilai dari grid di
  atas) -> data train punya sinyal cukup -> pola ini seharusnya bisa dipelajari lebih baik lewat
  augmentasi/durasi training, bukan ketiadaan data.
- **Similarity rendah ATAU gambar "tetangga" ternyata tidak nyambung sama sekali secara visual
  walau similarity-nya tinggi** -> data train tidak punya cakupan yang cukup untuk pola spesifik
  ini -> batasan data, bukan batasan teknik training.
- **Tetangga konsep/material-nya mirip TAPI labelnya beda-beda** -> ada inkonsistensi label untuk
  pola ini di data train sendiri.

Isi kesimpulan akhir di sini setelah melihat grid gambar di atas satu per satu.
